# This code is **`OUTDATED`**

Let's get the data from [FMI Weather](https://en.ilmatieteenlaitos.fi/observation-stations)

This is just a proof of concept that this can work

In [1]:
#!pip install fmiopendata

In [ ]:
import pandas as pd
from fmiopendata.wfs import download_stored_query
from datetime import datetime, timedelta
import time

import numpy as np

## Dataset

I have been researching to find the best stations across Finland's cities that would offer the best quality of data, but most of the observation center would either have missing features (`precipitation`, or `wind speed`).

So we will get multiple centers to compensate for this and merge them later into cities

Also we are getting the dataset from **Oulu, Helsinki, Tampere, Rovaniemi, and Vaasa** within the last 30 days, and we are also adding in the `latitude` and `longtitude` for each cities


Data was pulled from `02/03/2026`

In [ ]:
reliable_stations = {
    "101786": {"city": "Oulu", "name": "Oulu Airport", "lat": 64.93, "lon": 25.35},
    "101004": {"city": "Helsinki", "name": "Helsinki Kumpula", "lat": 60.20, "lon": 24.96},
    "101118": {"city": "Tampere", "name": "Tampere Airport", "lat": 61.41, "lon": 23.60},
    "101124": {"city": "Tampere", "name": "Tampere Harmala", "lat": 61.47, "lon": 23.75},
    "100949": {"city": "Turku", "name": "Turku Artukainen", "lat": 60.51, "lon": 22.20},
    "101933": {"city": "Rovaniemi", "name": "Rovaniemi Airport", "lat": 66.56, "lon": 25.83},
    "101485": {"city": "Vaasa", "name": "Vaasa Klemettila", "lat": 63.09, "lon": 21.65},
    "101462": {"city": "Vaasa", "name": "Vaasa Airport", "lat": 63.05, "lon": 21.76}
}

now = datetime.now()
chunks = [
    (now - timedelta(days=30), now),              # Last 30 days
    (now - timedelta(days=60), now - timedelta(days=31)), # 31-60 days ago
    (now - timedelta(days=90), now - timedelta(days=61))  # 61-90 days ago
]

all_chunks_dfs = []

for start, end in chunks:

    starttime_str = start.strftime('%Y-%m-%dT%H:%M:%SZ')
    endtime_str = end.strftime('%Y-%m-%dT%H:%M:%SZ')

    all_dfs = []
    print(f"\n--- Fetching Window: {starttime_str} to {endtime_str} ---")
    for fmisid, info in reliable_stations.items():
        print(f"Fetching {info['name']} (ID: {fmisid})...")
        try:
            obs = download_stored_query("fmi::observations::weather::hourly::multipointcoverage",
                                        args=[f"fmisid={fmisid}", 
                                            f"starttime={starttime_str}", 
                                            f"endtime={endtime_str}"])
            if obs.data:
                rows = []
                for timestamp, stations in obs.data.items():
                    for station_name, parameters in stations.items():
                        # Inject metadata directly into each row
                        row = {
                            'time': timestamp,
                            'city_group': info['city'], # Use this for merging
                            'station_name': info['name'],
                            'lat': info['lat'],
                            'lon': info['lon']
                        }
                        for p_name, p_data in parameters.items():
                            row[p_name.replace(' ', '_').lower()] = p_data.get('value')
                        rows.append(row)
                
                all_dfs.append(pd.DataFrame(rows))
                time.sleep(4) # API Politeness
        except Exception as e:
            print(f"Error at {info['name']}: {e}")

    # Create the Raw Master DF
    print("Concatting..")
    df_raw = pd.concat(all_dfs, ignore_index=True)
    all_chunks_dfs.append(df_raw)

df_90_days = pd.concat(all_chunks_dfs, ignore_index=True)
print("Complete")


--- Fetching Window: 2026-01-31T13:14:11Z to 2026-03-02T13:14:11Z ---
Fetching Oulu Airport (ID: 101786)...
Fetching Helsinki Kumpula (ID: 101004)...
Fetching Tampere Airport (ID: 101118)...
Fetching Tampere Harmala (ID: 101124)...
Fetching Turku Artukainen (ID: 100949)...
Fetching Rovaniemi Airport (ID: 101933)...
Fetching Vaasa Klemettila (ID: 101485)...
Fetching Vaasa Airport (ID: 101462)...
Concatting..

--- Fetching Window: 2026-01-01T13:14:11Z to 2026-01-30T13:14:11Z ---
Fetching Oulu Airport (ID: 101786)...
Fetching Helsinki Kumpula (ID: 101004)...
Fetching Tampere Airport (ID: 101118)...
Fetching Tampere Harmala (ID: 101124)...
Fetching Turku Artukainen (ID: 100949)...
Fetching Rovaniemi Airport (ID: 101933)...
Fetching Vaasa Klemettila (ID: 101485)...
Fetching Vaasa Airport (ID: 101462)...
Concatting..

--- Fetching Window: 2025-12-02T13:14:11Z to 2025-12-31T13:14:11Z ---
Fetching Oulu Airport (ID: 101786)...
Fetching Helsinki Kumpula (ID: 101004)...
Fetching Tampere Airport 

In [3]:
df_90_days.to_csv('weather_data.csv', index = False)